In [1]:
# @launchit.collected

In [2]:
import sys
import os
import json
import re
import pprint
import multiprocessing as mp
from collections import namedtuple # @launchit.collect
import dataclasses
from dataclasses import dataclass 
import importlib.util
import IPython
from enum import StrEnum, auto

import numpy as np

import optuna
from optuna.trial import TrialState
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torchvision import datasets
from torchvision import transforms

project_root_path = '${PROJECT_ROOT_PATH}'
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(project_root_path)
import lib.launchit # @launchit.disable
import lib.optuna_multiprocessing 

In [3]:
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

RNG = np.random.default_rng()

CONFIG = namedtuple('Config', 
                    'project_root_path, project_root_uri, subproject_path, data_path, mnist_path, run_path, ' + 
                    'self_fname, self_name, ' +
                    'subproject_name,' +
                    'is_cuda, cuda_device, is_cuml, exec_mode')(
    project_root_path=project_root_path,
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    subproject_path=os.path.abspath('.'),
    data_path=os.path.join(project_root_path, 'data'),
    mnist_path=os.path.join(project_root_path, 'data', 'mnist'),
    run_path='',
    self_fname='',
    self_name='',
    subproject_name='',
    is_cuda=torch.cuda.is_available(),
    cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
    is_cuml=importlib.util.find_spec('cuml') is not None,
    exec_mode=ExecMode.MASTER_NOTEBOOK,
)

if IPython.get_ipython() is None:
    module_fname = __file__
    module_basename = os.path.basename(module_fname)
    module_name, _ = os.path.splitext(module_basename)
    
    CONFIG = CONFIG._replace(self_fname=module_fname, self_name=module_name)
    CONFIG = CONFIG._replace(exec_mode=ExecMode.LAUNCH_MODULE)
else:
    with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
        notebook_fname = json.load(cf)['jupyter_session']
        notebook_basename = os.path.basename(notebook_fname)
        notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    
        m = re.match(r'(\w+)-Copy\d+$', notebook_name)
    
        if m:
            # Cuml is used to be launched from the copy of the notebook
            notebook_name = m.group(1)

        CONFIG = CONFIG._replace(self_fname=notebook_fname, self_name=notebook_name)
        
        is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
        CONFIG = CONFIG._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)

CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.self_fname)))
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', 'experiment', CONFIG.subproject_name))
print('CONFIG=\n' + pprint.pformat(CONFIG._asdict()))
print('')

os.makedirs(CONFIG.run_path, exist_ok=True)

# @launchit.disable
# @launchit.collect
MODEL_INSTANCE_INFO = namedtuple('ModelInstanceInfo', 'group_uri, name, version, main_asset_fname')(
    group_uri='${MODEL_GROUP_URI}',
    name='${MODEL_NAME}',
    version='${MODEL_VERSION}',
    main_asset_fname='${LAUNCHIT_FNAME}'
)
# @launchit.stop

MODEL_INSTANCE_INFO = MODEL_INSTANCE_INFO._replace(group_uri=f'{CONFIG.project_root_uri}.experiment.{CONFIG.subproject_name}')
MODEL_INSTANCE_INFO = MODEL_INSTANCE_INFO._replace(name=CONFIG.self_name)
MODEL_INSTANCE_INFO = MODEL_INSTANCE_INFO._replace(version='')
MODEL_INSTANCE_INFO = MODEL_INSTANCE_INFO._replace(main_asset_fname=CONFIG.self_fname)
# @launchit.stop

print('MODEL_INSTANCE_INFO=\n' + pprint.pformat(MODEL_INSTANCE_INFO._asdict()))

CONFIG=
{'cuda_device': 'cuda',
 'data_path': '/home/misha/dev/mine/neurovision/data',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_cuda': True,
 'is_cuml': False,
 'mnist_path': '/home/misha/dev/mine/neurovision/data/mnist',
 'project_root_path': '/home/misha/dev/mine/neurovision',
 'project_root_uri': 'com.develorium.neurovision',
 'run_path': '/home/misha/dev/mine/neurovision/run/experiment/optuna',
 'self_fname': '/home/misha/dev/mine/neurovision/experiment/optuna/optuna_torch_multiproc_test2.ipynb',
 'self_name': 'optuna_torch_multiproc_test2',
 'subproject_name': 'optuna',
 'subproject_path': '/home/misha/dev/mine/neurovision/experiment/optuna'}

MODEL_INSTANCE_INFO=
{'group_uri': 'com.develorium.neurovision.experiment.optuna',
 'main_asset_fname': '/home/misha/dev/mine/neurovision/experiment/optuna/optuna_torch_multiproc_test2.ipynb',
 'name': 'optuna_torch_multiproc_test2',
 'version': ''}


In [4]:
BATCHSIZE = 128
CLASSES = 10
EPOCHS = 10
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10

In [5]:
def define_model(trial):
    # We optimize the number of layers, hidden units and dropout ratio in each layer.
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []

    in_features = 28 * 28
    for i in range(n_layers):
        out_features = trial.suggest_int("n_units_l{}".format(i), 4, 128)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())
        p = trial.suggest_float("dropout_l{}".format(i), 0.2, 0.5)
        layers.append(nn.Dropout(p))

        in_features = out_features
    layers.append(nn.Linear(in_features, CLASSES))
    layers.append(nn.LogSoftmax(dim=1))

    return nn.Sequential(*layers)

In [6]:
def get_mnist():
    # Load FashionMNIST dataset.
    train_loader = torch.utils.data.DataLoader(
        datasets.FashionMNIST(CONFIG.mnist_path, train=True, download=True, transform=transforms.ToTensor()),
        batch_size=BATCHSIZE,
        shuffle=True,
    )
    valid_loader = torch.utils.data.DataLoader(
        datasets.FashionMNIST(CONFIG.mnist_path, train=False, transform=transforms.ToTensor()),
        batch_size=BATCHSIZE,
        shuffle=True,
    )

    return train_loader, valid_loader

In [7]:
if CONFIG.exec_mode == ExecMode.LAUNCH_MODULE:
    trial = lib.optuna_multiprocessing.get_trial()
    
    # Generate the model.
    model = define_model(trial).to(CONFIG.cuda_device)
    
    # Generate the optimizers.
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    
    # Get the FashionMNIST dataset.
    train_loader, valid_loader = get_mnist()
    
    # Training of the model.
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            # Limiting training data for faster epochs.
            if batch_idx * BATCHSIZE >= N_TRAIN_EXAMPLES:
                break
    
            data, target = data.view(data.size(0), -1).to(CONFIG.cuda_device), target.to(CONFIG.cuda_device)
    
            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()
    
        # Validation of the model.
        model.eval()
        correct = 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(valid_loader):
                # Limiting validation data.
                if batch_idx * BATCHSIZE >= N_VALID_EXAMPLES:
                    break
                data, target = data.view(data.size(0), -1).to(CONFIG.cuda_device), target.to(CONFIG.cuda_device)
                output = model(data)
                # Get the index of the max log-probability.
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()
    
        accuracy = correct / min(len(valid_loader.dataset), N_VALID_EXAMPLES)
    
        trial.report(accuracy, epoch)
    
        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    
    lib.optuna_multiprocessing.save_trial_result(accuracy)

In [8]:
# @launchit.disable
expandvars = dict(
    PROJECT_ROOT_PATH = project_root_path,
    SUBPROJECT_PATH = CONFIG.subproject_path,
    MODEL_GROUP_URI=MODEL_INSTANCE_INFO.group_uri,
    MODEL_NAME=MODEL_INSTANCE_INFO.name,
)

mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch
rop_params = lib.optuna_multiprocessing.RunOptimizationParameters(
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=MODEL_INSTANCE_INFO.group_uri,
    model_name=MODEL_INSTANCE_INFO.name,
    expandvars=expandvars,
    run_path=CONFIG.run_path,
    study_name=CONFIG.self_name,
    study_fname=os.path.join(CONFIG.run_path, CONFIG.self_name + '.log'),
    collect_inds=None,
    verbosity=2,
)

with mp_ctx.Pool(processes=4) as pool:
    rop_list = [rop_params] * 10
    pool.map(lib.optuna_multiprocessing.run_optimization, rop_list)

# join and report
study = optuna.create_study(
    study_name=rop_params.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_params.study_fname)),
    load_if_exists=True, # Useful for multi-process or multi-node optimization.
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

Found com.develorium.neurovision.experiment.optuna.optuna_torch_multiproc_test2:19, continuing series with version=20
Model instance registered, version=20
Creating /home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch20.py
Created "/home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch20.py"


[I 2026-01-25 15:24:07,073] Using an existing study with name 'optuna_torch_multiproc_test2' instead of creating a new one.


Found com.develorium.neurovision.experiment.optuna.optuna_torch_multiproc_test2:19, continuing series with version=20
Got Bad Requets status (race condition?), retrying in 1 seconds
Found com.develorium.neurovision.experiment.optuna.optuna_torch_multiproc_test2:20, continuing series with version=21
Model instance registered, version=21
Creating /home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch21.py
Created "/home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch21.py"


[I 2026-01-25 15:24:08,127] Using an existing study with name 'optuna_torch_multiproc_test2' instead of creating a new one.


Found com.develorium.neurovision.experiment.optuna.optuna_torch_multiproc_test2:19, continuing series with version=20
Got Bad Requets status (race condition?), retrying in 1 seconds
Found com.develorium.neurovision.experiment.optuna.optuna_torch_multiproc_test2:20, continuing series with version=21
Got Bad Requets status (race condition?), retrying in 2 seconds
Found com.develorium.neurovision.experiment.optuna.optuna_torch_multiproc_test2:21, continuing series with version=22
Model instance registered, version=22
Creating /home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch22.py
Created "/home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch22.py"
Found com.develorium.neurovision.experiment.optuna.optuna_torch_multiproc_test2:19, continuing series with version=20
Got Bad Requets status (race condition?), retrying in 1 seconds
Found com.develorium.neurovision.experiment.optuna.optuna_torch_multiproc_test2:20, con

...

CONFIG=
{'cuda_device': 'cuda',
 'data_path': '/home/misha/dev/mine/neurovision/data',
 'exec_mode': <ExecMode.LAUNCH_MODULE: 'launch_module'>,
 'is_cuda': True,
 'is_cuml': False,
 'mnist_path': '/home/misha/dev/mine/neurovision/data/mnist',
 'project_root_path': '/home/misha/dev/mine/neurovision',
 'project_root_uri': 'com.develorium.neurovision',
 'run_path': '/home/misha/dev/mine/neurovision/run/experiment/optuna',
 'self_fname': '/home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch24.py',
 'self_name': 'optuna_torch_multiproc_test2-launch24',
 'subproject_name': 'optuna',
 'subproject_path': '/home/misha/dev/mine/neurovision/experiment/optuna'}

MODEL_INSTANCE_INFO=
{'group_uri': 'com.develorium.neurovision.experiment.optuna',
 'main_asset_fname': '/home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch24.py',
 'name': 'optuna_torch_multiproc_test2',
 'version': '24'}
Finished optuna_torch_multiproc_test2-lau

[I 2026-01-25 15:24:27,344] Trial 29 finished with value: 0.828125 and parameters: {'n_layers': 2, 'n_units_l0': 110, 'dropout_l0': 0.44526820668227557, 'n_units_l1': 101, 'dropout_l1': 0.37472012682823946, 'optimizer': 'Adam', 'lr': 0.0032929853058338886}. Best is trial 29 with value: 0.828125.
[I 2026-01-25 15:24:27,428] Using an existing study with name 'optuna_torch_multiproc_test2' instead of creating a new one.
[I 2026-01-25 15:24:27,937] Trial 32 pruned. 
[I 2026-01-25 15:24:28,171] Trial 30 pruned. 
[I 2026-01-25 15:24:28,517] Trial 33 pruned. 


CONFIG=
{'cuda_device': 'cuda',
 'data_path': '/home/misha/dev/mine/neurovision/data',
 'exec_mode': <ExecMode.LAUNCH_MODULE: 'launch_module'>,
 'is_cuda': True,
 'is_cuml': False,
 'mnist_path': '/home/misha/dev/mine/neurovision/data/mnist',
 'project_root_path': '/home/misha/dev/mine/neurovision',
 'project_root_uri': 'com.develorium.neurovision',
 'run_path': '/home/misha/dev/mine/neurovision/run/experiment/optuna',
 'self_fname': '/home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch26.py',
 'self_name': 'optuna_torch_multiproc_test2-launch26',
 'subproject_name': 'optuna',
 'subproject_path': '/home/misha/dev/mine/neurovision/experiment/optuna'}

MODEL_INSTANCE_INFO=
{'group_uri': 'com.develorium.neurovision.experiment.optuna',
 'main_asset_fname': '/home/misha/dev/mine/neurovision/run/experiment/optuna/optuna_torch_multiproc_test2-launch26.py',
 'name': 'optuna_torch_multiproc_test2',
 'version': '26'}
Finished optuna_torch_multiproc_test2-lau

[I 2026-01-25 15:24:31,037] Trial 31 finished with value: 0.64921875 and parameters: {'n_layers': 3, 'n_units_l0': 77, 'dropout_l0': 0.31969057519482835, 'n_units_l1': 13, 'dropout_l1': 0.24943568310391434, 'n_units_l2': 105, 'dropout_l2': 0.48909197949212224, 'optimizer': 'RMSprop', 'lr': 0.00030380172085591546}. Best is trial 29 with value: 0.828125.


Exception: Failed to generate new launch file name: all variants are taken